# Checkpoint 5 — first independent XGBoost models

This checkpoint builds one model for each horizon using the exact horizon CSVs from notebook 01 and the exact channel-grouped assignments from notebook 04.

The notebook performs:

1. feature-contract checks;
2. a visible fold-safe preprocessing example;
3. five-fold grouped cross-validation for every horizon;
4. comparison with three training-only baselines;
5. fitting and saving one candidate XGBoost bundle per horizon;
6. tests of the saved predictions and model artifacts.

The reserved test partition is not transformed, predicted, or evaluated here. Hyperparameter tuning is also left for the next checkpoint.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_checkpoint5_models import (
    HORIZONS,
    MODEL_NAME,
    load_horizon_checkpoint,
    train_all_horizons,
)
from viewcastlk_ml.horizon_preprocessing import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    ENGINEERED_NUMERIC_COLUMNS,
    EXCLUDED_MODEL_COLUMNS,
    LLM_SCORE_COLUMNS,
    RAW_NUMERIC_COLUMNS,
    TOPIC_COLUMNS,
    HorizonDatasetPreprocessor,
)

ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint5_xgboost'

feature_group_summary = pd.DataFrame([
    {'group': 'raw numeric', 'count': len(RAW_NUMERIC_COLUMNS), 'features': ', '.join(RAW_NUMERIC_COLUMNS)},
    {'group': 'engineered numeric', 'count': len(ENGINEERED_NUMERIC_COLUMNS), 'features': ', '.join(ENGINEERED_NUMERIC_COLUMNS)},
    {'group': 'boolean and topic', 'count': len(BOOLEAN_COLUMNS), 'features': ', '.join(BOOLEAN_COLUMNS)},
    {'group': 'one-hot categorical', 'count': len(CATEGORICAL_COLUMNS), 'features': ', '.join(CATEGORICAL_COLUMNS)},
    {'group': 'target encoded', 'count': 1, 'features': 'category_name'},
    {'group': 'LLM scores reserved, currently disabled', 'count': len(LLM_SCORE_COLUMNS), 'features': ', '.join(LLM_SCORE_COLUMNS)},
])
display(feature_group_summary)

loaded = {}
inventory_rows = []
for horizon in HORIZONS:
    X, y, assignments, data_path, assignment_path = load_horizon_checkpoint(PROJECT_ROOT, horizon)
    loaded[horizon] = (X, y, assignments)
    inventory_rows.append({
        'horizon_days': horizon,
        'rows': len(X),
        'source_predictor_columns': X.shape[1],
        'development_rows': int(assignments['partition'].eq('development').sum()),
        'reserved_test_rows': int(assignments['partition'].eq('test_reserved').sum()),
        'channel_overlap': len(
            set(assignments.loc[assignments['partition'].eq('development'), 'channel_id'].astype(str))
            & set(assignments.loc[assignments['partition'].eq('test_reserved'), 'channel_id'].astype(str))
        ),
    })
inventory = pd.DataFrame(inventory_rows)
display(inventory)

                                     group  ...                                           features
0                              raw numeric  ...  duration_seconds, ch_subs_at_publish, ch_video...
1                       engineered numeric  ...                                  ch_videos_per_day
2                        boolean and topic  ...  is_short, made_for_kids, publish_is_weekend, t...
3                      one-hot categorical  ...              default_language, publish_time_bucket
4                           target encoded  ...                                      category_name
5  LLM scores reserved, currently disabled  ...  title_urgency, title_emotional_appeal, title_s...

[6 rows x 3 columns]
   horizon_days   rows  ...  reserved_test_rows  channel_overlap
0             7  20663  ...                4115                0
1            14  15685  ...                3137                0
2            21  15100  ...                3028                0
3            30  14753  .

In [2]:
# Fit the preprocessor on day-7 fold-1 training rows only, then show real transformed output.
X7, y7, assignments7 = loaded[7]
fold1_train_mask = (
    assignments7['partition'].eq('development')
    & ~assignments7['cv_validation_fold'].eq(1)
)
fold1_validation_mask = (
    assignments7['partition'].eq('development')
    & assignments7['cv_validation_fold'].eq(1)
)
train_positions = assignments7.loc[fold1_train_mask, 'horizon_row_position'].astype(int).to_numpy()
validation_positions = assignments7.loc[fold1_validation_mask, 'horizon_row_position'].astype(int).to_numpy()

preview_preprocessor = HorizonDatasetPreprocessor(
    category_smoothing=10.0,
    include_llm_scores=False,
)
preview_preprocessor.fit(X7.iloc[train_positions], np.log1p(y7.iloc[train_positions]))
transformed_preview = preview_preprocessor.transform(X7.iloc[validation_positions[:5]])

print(f'Transformed model width: {transformed_preview.shape[1]} numeric features')
print('First 15 ordered feature names:')
print(preview_preprocessor.get_feature_names_out()[:15].tolist())
display(transformed_preview.iloc[:, :15])

unseen_example = X7.iloc[[validation_positions[0]]].copy()
unseen_example['category_name'] = 'CATEGORY_NOT_PRESENT_IN_TRAINING'
unseen_value = preview_preprocessor.transform(unseen_example)['category_encoded_log'].iloc[0]

preprocessing_tests = pd.DataFrame([
    {'test': 'output is entirely numeric', 'passed': all(np.issubdtype(dtype, np.number) for dtype in transformed_preview.dtypes)},
    {'test': 'no infinity produced', 'passed': not np.isinf(transformed_preview.to_numpy()).any()},
    {'test': 'category raw value is not a model feature', 'passed': 'category_name' not in transformed_preview.columns},
    {'test': 'smoothed category encoding exists', 'passed': 'category_encoded_log' in transformed_preview.columns},
    {'test': 'all 19 topic flags are retained', 'passed': set(TOPIC_COLUMNS).issubset(transformed_preview.columns)},
    {'test': 'removed columns are blocked', 'passed': set(EXCLUDED_MODEL_COLUMNS).isdisjoint(transformed_preview.columns)},
    {'test': 'LLM columns are disabled for this run', 'passed': set(LLM_SCORE_COLUMNS).isdisjoint(transformed_preview.columns)},
    {'test': 'unseen category uses training global mean', 'passed': np.isclose(unseen_value, preview_preprocessor.category_encoder_.global_mean_)},
    {'test': 'preprocessor fitted on fold training rows only', 'passed': len(train_positions) + len(validation_positions) == int(assignments7['partition'].eq('development').sum())},
])
preprocessing_tests['status'] = np.where(preprocessing_tests['passed'], 'PASS', 'FAIL')
display(preprocessing_tests[['test', 'status']])
assert preprocessing_tests['passed'].all(), preprocessing_tests.loc[~preprocessing_tests['passed']]

Transformed model width: 56 numeric features
First 15 ordered feature names:
['duration_seconds', 'ch_subs_at_publish', 'ch_videos_at_publish', 'channel_age_days_at_publish', 'ch_videos_per_day', 'is_short', 'made_for_kids', 'publish_is_weekend', 'topic_entertainment', 'topic_fashion', 'topic_food', 'topic_gaming', 'topic_health', 'topic_hobby', 'topic_humour']
    duration_seconds  ch_subs_at_publish  ...  topic_hobby  topic_humour
0              192.0           3770000.0  ...          0.0           0.0
1              422.0           3770000.0  ...          0.0           0.0
8              992.0           3780000.0  ...          0.0           0.0
9             2568.0              7340.0  ...          0.0           0.0
12             155.0           3770000.0  ...          0.0           0.0

[5 rows x 15 columns]
                                             test status
0                      output is entirely numeric   PASS
1                            no infinity produced   PASS
2   

In [3]:
# This is the actual training run. It performs 5 folds x 4 horizons, then saves 4 bundles.
training_run = train_all_horizons(
    project_root=PROJECT_ROOT,
    output_dir=ARTIFACT_DIR,
    n_estimators=800,
    n_jobs=4,
    include_llm_scores=False,
)

summary = training_run['summary']
display(summary[[
    'horizon_days', 'model', 'rows', 'mape_nonzero_pct',
    'median_ape_nonzero_pct', 'smape_pct', 'rmsle', 'log_r2'
]])


Training independent day-7 model
day 7 fold 1/5: RMSLE=1.8674, median APE=105.31%, best trees=103
day 7 fold 2/5: RMSLE=2.1913, median APE=85.55%, best trees=194
day 7 fold 3/5: RMSLE=2.1813, median APE=94.08%, best trees=91
day 7 fold 4/5: RMSLE=1.8448, median APE=84.98%, best trees=73
day 7 fold 5/5: RMSLE=2.2375, median APE=99.32%, best trees=41

Training independent day-14 model
day 14 fold 1/5: RMSLE=1.9254, median APE=141.67%, best trees=33
day 14 fold 2/5: RMSLE=2.2086, median APE=90.81%, best trees=1
day 14 fold 3/5: RMSLE=2.4852, median APE=98.68%, best trees=339
day 14 fold 4/5: RMSLE=1.9077, median APE=89.75%, best trees=135
day 14 fold 5/5: RMSLE=1.9881, median APE=91.06%, best trees=125

Training independent day-21 model
day 21 fold 1/5: RMSLE=1.9365, median APE=116.94%, best trees=107
day 21 fold 2/5: RMSLE=1.9060, median APE=84.94%, best trees=104
day 21 fold 3/5: RMSLE=1.9588, median APE=89.33%, best trees=116
day 21 fold 4/5: RMSLE=1.9748, median APE=89.50%, best tree

In [4]:
# Compare XGBoost with the strongest baseline and show the most-used features.
comparison_rows = []
for horizon in [str(h) for h in HORIZONS] + ['combined']:
    group = summary[summary['horizon_days'].astype(str).eq(horizon)].set_index('model')
    baseline = group.loc['category_tier_median']
    model = group.loc[MODEL_NAME]
    comparison_rows.append({
        'horizon_days': horizon,
        'baseline_rmsle': baseline['rmsle'],
        'xgboost_rmsle': model['rmsle'],
        'rmsle_improvement_pct': 100 * (baseline['rmsle'] - model['rmsle']) / baseline['rmsle'],
        'baseline_mape_pct': baseline['mape_nonzero_pct'],
        'xgboost_mape_pct': model['mape_nonzero_pct'],
        'mape_improvement_pct': 100 * (baseline['mape_nonzero_pct'] - model['mape_nonzero_pct']) / baseline['mape_nonzero_pct'],
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison)

importance = training_run['feature_importance']
top_features = (
    importance.sort_values(['horizon_days', 'gain_share_within_horizon'], ascending=[True, False])
    .groupby('horizon_days', as_index=False)
    .head(10)
)
display(top_features[['horizon_days', 'feature', 'gain_share_within_horizon']])

  horizon_days  baseline_rmsle  ...  xgboost_mape_pct  mape_improvement_pct
0            7        2.304437  ...        656.848544             35.253191
1           14        2.350057  ...        880.984571             15.975579
2           21        2.326375  ...        735.768853             28.088663
3           30        2.295737  ...        947.673503              4.677062
4     combined        2.318397  ...        792.726547             22.280928

[5 rows x 7 columns]
     horizon_days                            feature  gain_share_within_horizon
19              7                     topic_politics                   0.374331
1               7                 ch_subs_at_publish                   0.096244
5               7                           is_short                   0.032150
0               7                   duration_seconds                   0.030364
2               7               ch_videos_at_publish                   0.029573
3               7        channel_age_days_

In [5]:
# Validate the serialized artifacts without touching the reserved test rows.
manifest = json.loads((ARTIFACT_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(ARTIFACT_DIR / 'cv_predictions.csv')
artifact_tests = []

def record_test(name, condition, detail=''):
    artifact_tests.append({
        'test': name,
        'status': 'PASS' if bool(condition) else 'FAIL',
        'detail': detail,
    })

record_test('manifest marks reserved test unevaluated', manifest['status'] == 'candidate_reserved_test_not_evaluated')
record_test('four independent model records', {m['horizon_days'] for m in manifest['models']} == set(HORIZONS))
record_test('LLM scores disabled until complete backfill', manifest['llm_scores_enabled'] is False)
record_test('CV predictions are finite', np.isfinite(predictions.filter(like='predicted_').to_numpy(dtype=float)).all())
record_test('CV predictions are non-negative', (predictions.filter(like='predicted_').to_numpy(dtype=float) >= 0).all())

for model_record in manifest['models']:
    horizon = model_record['horizon_days']
    bundle = joblib.load(ARTIFACT_DIR / model_record['model_path'])
    feature_order = json.loads((ARTIFACT_DIR / model_record['feature_order_path']).read_text(encoding='utf-8'))
    X, y, assignments = loaded[horizon]
    development_position = int(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].iloc[0])
    sample_prediction = bundle.predict_views(X.iloc[[development_position]])
    development_positions = set(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].astype(int))
    test_positions = set(assignments.loc[assignments['partition'].eq('test_reserved'), 'horizon_row_position'].astype(int))
    predicted_positions = set(predictions.loc[predictions['horizon_days'].eq(horizon), 'horizon_row_position'].astype(int))

    record_test(f'day {horizon} bundle loads and predicts', len(sample_prediction) == 1 and np.isfinite(sample_prediction).all() and (sample_prediction >= 0).all())
    record_test(f'day {horizon} feature order matches bundle', feature_order == bundle.feature_names)
    record_test(f'day {horizon} every development row has one OOF prediction', predicted_positions == development_positions)
    record_test(f'day {horizon} reserved test has no prediction', predicted_positions.isdisjoint(test_positions))
    record_test(f'day {horizon} removed features absent', set(EXCLUDED_MODEL_COLUMNS).isdisjoint(feature_order))

artifact_tests = pd.DataFrame(artifact_tests)
display(artifact_tests)
failures = artifact_tests[artifact_tests['status'].eq('FAIL')]
assert failures.empty, failures.to_string(index=False)
print(f'PASS: all {len(preprocessing_tests) + len(artifact_tests)} notebook checks succeeded.')

                                                 test status detail
0            manifest marks reserved test unevaluated   PASS       
1                      four independent model records   PASS       
2         LLM scores disabled until complete backfill   PASS       
3                           CV predictions are finite   PASS       
4                     CV predictions are non-negative   PASS       
5                     day 7 bundle loads and predicts   PASS       
6                  day 7 feature order matches bundle   PASS       
7   day 7 every development row has one OOF predic...   PASS       
8               day 7 reserved test has no prediction   PASS       
9                       day 7 removed features absent   PASS       
10                   day 14 bundle loads and predicts   PASS       
11                day 14 feature order matches bundle   PASS       
12  day 14 every development row has one OOF predi...   PASS       
13             day 14 reserved test has no predi

## Checkpoint result

The first four candidate XGBoost models now exist and beat the strongest category-by-channel-tier baseline in cross-validation. Their absolute error is still too high for a final product claim, so these are pipeline candidates rather than frozen production models.

The next checkpoint is controlled hyperparameter tuning on the same saved development folds. The reserved test remains untouched until the final selected configuration is frozen.